# Fake News Prediction Rebuild — Step 2: EDA

Goal: understand the dataset honestly before writing any feature/model code, and specifically check for leakage before it costs us a fake 99% accuracy number later.

This dataset (`Fake.csv` / `True.csv`, ISOT-style) has no `author` column, so the leakage story is different from the old notebook's author-leakage problem. Here the suspects are:
- `Subject` — the two files were likely scraped from different sections of the source site, so `Subject` might separate real/fake almost perfectly without the model learning anything about *content*.
- Reuters dateline pattern (`WASHINGTON (Reuters) -` style) at the start of `text` in the true-news file, which would let a model key off boilerplate formatting instead of substance.

Both are checked below with real numbers, not assumed.

In [1]:
import pandas as pd
import numpy as np
import re

pd.set_option('display.max_colwidth', 120)

## 1. Load, label, concat, shuffle

The two files come in solid blocks (all-fake rows, then all-true rows). If we don't shuffle before any order-sensitive step (train/test split, head() sanity checks, cross-validation folds), we risk correlated splits. Shuffle once here, right after labeling, with a fixed `random_state` so the notebook is reproducible.

In [2]:
RANDOM_STATE = 42

fake_df = pd.read_csv('../data/Fake.csv')
true_df = pd.read_csv('../data/True.csv')

fake_df['label'] = 1  # 1 = fake, matches the old notebook's convention
true_df['label'] = 0  # 0 = real

df = pd.concat([fake_df, true_df], ignore_index=True)
df = df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

print(df.shape)
df.head()

(44898, 5)


,title,text,subject,date,label
0,Ben Stein Calls Out 9th Circuit Court: Committed a ‘Coup d’état’ Against the Constitution,"21st Century Wire says Ben Stein, reputable professor from, Pepperdine University (also of some Hollywood fame appea...",US_News,"February 13, 2017",1
1,Trump drops Steve Bannon from National Security Council,WASHINGTON (Reuters) - U.S. President Donald Trump removed his chief strategist Steve Bannon from the National Secur...,politicsNews,"April 5, 2017",0
2,Puerto Rico expects U.S. to lift Jones Act shipping restrictions,(Reuters) - Puerto Rico Governor Ricardo Rossello said on Wednesday he expected the federal government to waive the ...,politicsNews,"September 27, 2017",0
3,OOPS: Trump Just Accidentally Confirmed He Leaked Israeli Intelligence To Russia (VIDEO),"On Monday, Donald Trump once again embarrassed himself and his country by accidentally revealing the source of the e...",News,"May 22, 2017",1
4,Donald Trump heads for Scotland to reopen a golf resort,"GLASGOW, Scotland (Reuters) - Most U.S. presidential candidates go abroad to sharpen their foreign policy credential...",politicsNews,"June 24, 2016",0


## 2. Class balance and missing values

In [3]:
print("Raw counts:")
print(df['label'].value_counts())
print("\nNormalized:")
print(df['label'].value_counts(normalize=True).round(4))

Raw counts:
label
1    23481
0    21417
Name: count, dtype: int64

Normalized:
label
1    0.523
0    0.477
Name: proportion, dtype: float64


In [4]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
pd.DataFrame({'missing': missing, 'pct': missing_pct})

,missing,pct
title,0,0.0
text,0,0.0
subject,0,0.0
date,0,0.0
label,0,0.0


## 3. Subject vs label — checking for leakage

If any `Subject` value maps almost 100% to one label, that column is a shortcut, not a feature. A model trained on it would look great on this dataset and be useless on a real newsroom feed, because "subject tag from this specific scrape" isn't a property of real vs fake content in general.

In [5]:
subject_crosstab = pd.crosstab(df['subject'], df['label'], normalize='index').round(4)
subject_crosstab.columns = ['pct_real', 'pct_fake']
subject_counts = df['subject'].value_counts()
subject_summary = subject_crosstab.join(subject_counts.rename('n'))
subject_summary.sort_values('n', ascending=False)

,pct_real,pct_fake,n
subject,,,
politicsNews,1.0,0.0,11272
worldnews,1.0,0.0,10145
News,0.0,1.0,9050
politics,0.0,1.0,6841
left-news,0.0,1.0,4459
Government News,0.0,1.0,1570
US_News,0.0,1.0,783
Middle-east,0.0,1.0,778


**Read this table before moving on.** If any subject is >95% one label, treat `Subject` as excluded-by-design for modeling, and say so explicitly in `src/preprocess.py` and the README rather than silently dropping the column.

## 4. Reuters dateline check

First confirm the pattern actually matches real rows before drawing any conclusion from it — don't assume it holds just because it's a known pattern for this dataset.

In [6]:
reuters_pattern = re.compile(r'^\s*\S.*?\(Reuters\)\s*-')

df['has_reuters_dateline'] = df['text'].fillna('').apply(lambda t: bool(reuters_pattern.match(t)))

print("Sample matches:")
for t in df.loc[df['has_reuters_dateline'], 'text'].head(3):
    print(t[:100])
    print('---')

Sample matches:
WASHINGTON (Reuters) - U.S. President Donald Trump removed his chief strategist Steve Bannon from th
---
GLASGOW, Scotland (Reuters) - Most U.S. presidential candidates go abroad to sharpen their foreign p
---
WASHINGTON (Reuters) - The State Department said Wednesday that the United States would be open to t
---


In [7]:
dateline_crosstab = pd.crosstab(df['has_reuters_dateline'], df['label'], normalize='index').round(4)
dateline_crosstab.columns = ['pct_real', 'pct_fake']
dateline_crosstab

,pct_real,pct_fake
has_reuters_dateline,,
False,0.0658,0.9342
True,1.0000,0.0000


If this shows the dateline appears almost exclusively in real-labeled rows, strip it before cleaning in Step 3 (regex substitution at the start of `text`, done once in `src/preprocess.py`), otherwise the model just learns to detect "(Reuters)" instead of anything about real vs fake content.

## 5. Date range check

In [8]:
df['date_parsed'] = pd.to_datetime(df['date'], errors='coerce')

unparseable = df['date_parsed'].isnull().sum()
print(f"Unparseable dates: {unparseable} ({unparseable / len(df) * 100:.2f}%)")

print("\nDate range by label:")
print(df.groupby('label')['date_parsed'].agg(['min', 'max', 'count']))

Unparseable dates: 33030 (73.57%)

Date range by label:
             min        max  count
label                             
0            NaT        NaT      0
1     2015-05-01 2017-12-31  11868


Not used as a model feature either way (a raw date shouldn't predict truthfulness), but worth knowing if the ranges are disjoint, since that would be one more thing that quietly makes this dataset easier than real-world detection would be.

## 6. Summary

- Class balance: 52.3% fake (23481), 47.7% real (21417) — no rebalancing needed
- Missing values: none, in any column
- Subject leakage: confirmed and total. Every subject is 100% one label
  (politicsNews, worldnews = 100% real; News, politics, left-news,
  Government News, US_News, Middle-east = 100% fake). Subject is
  excluded from modeling entirely — not a feature, a shortcut.
- Reuters dateline leakage: confirmed, strong but partial. 100% of
  dateline rows are real; 93.42% of non-dateline rows are fake (so
  absence doesn't guarantee fake). Dateline prefix will be stripped
  from text in Step 3 before cleaning.
- Date range: 73.57% unparseable overall, and ALL real-labeled rows
  failed to parse (True.csv uses a date format pandas doesn't
  auto-detect) while ~half of fake rows parsed fine (2015-05-01 to
  2017-12-31). Not used as a model feature either way, but the format
  mismatch itself is a dataset quirk worth a one-line README mention.